[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/13_svm_and_kernels/01_maximum_margin_intuition.ipynb)

# Bài 01 — SVM: ranh giới có lề lớn nhất

**🎯 Mục tiêu**
- Câu hỏi: dữ liệu tách được tuyến tính có **vô số** đường phân tách — chọn đường nào?
- Định nghĩa **lề (margin)** và hình học khoảng cách từ điểm tới siêu phẳng.
- Bài toán **hard-margin SVM**: $\min \frac12\lVert\mathbf w\rVert^2$ s.t. $y_i(\mathbf w^\top\mathbf x_i + b) \ge 1$.
- **Support vectors**: chỉ vài điểm quyết định ranh giới. Giải bài toán bằng `scipy.optimize` và đối chiếu với scikit-learn.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/13_svm_and_kernels

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.svm import SVC

from ml_course.visualization import PALETTE, set_style

set_style()
rng = np.random.default_rng(3)

## 1. Vô số đường phân tách — đường nào tốt nhất?

In [ ]:
X = np.vstack([rng.normal([-1.6, -1.2], 0.55, (25, 2)), rng.normal([1.6, 1.2], 0.55, (25, 2))])
y = np.r_[-np.ones(25), np.ones(25)]


def draw_points(ax):
    for c, label in [(-1, "lớp −1"), (1, "lớp +1")]:
        ax.scatter(X[y == c, 0], X[y == c, 1], color=PALETTE[int(c > 0)], s=35, label=label, zorder=3)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-3.5, 3.5)
    ax.set_aspect("equal")


xs = np.linspace(-4, 4, 50)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
draw_points(ax)
for seed in range(6):
    p = Perceptron(random_state=seed, shuffle=True).fit(X, y)
    w, b = p.coef_[0], p.intercept_[0]
    ax.plot(xs, -(w[0] * xs + b) / w[1], lw=1, alpha=0.7)
ax.set_title("Perceptron với 6 seed: 6 đường đều tách đúng 100%")
plt.show()

Mọi đường trên đều có lỗi train = 0, nhưng một số đi **sát** các điểm — điểm mới nhiễu nhẹ là bị phân loại sai. Trực giác: chọn đường cách **xa nhất** các điểm của cả hai lớp → "đường giữa con đường rộng nhất".

## 2. Hình học: khoảng cách tới siêu phẳng

Siêu phẳng $\mathbf w^\top\mathbf x + b = 0$. Khoảng cách (có dấu) từ điểm $\mathbf x_i$:

$$d_i = \frac{\mathbf w^\top\mathbf x_i + b}{\lVert\mathbf w\rVert}$$

**Margin** của bộ phân loại = khoảng cách từ siêu phẳng tới điểm **gần nhất**: $\min_i \frac{y_i(\mathbf w^\top\mathbf x_i + b)}{\lVert\mathbf w\rVert}$.

Vì nhân $(\mathbf w, b)$ với hằng số không đổi siêu phẳng, ta **chuẩn hoá** sao cho điểm gần nhất có $y_i(\mathbf w^\top\mathbf x_i + b) = 1$ → margin $= 1/\lVert\mathbf w\rVert$, độ rộng "con đường" $= 2/\lVert\mathbf w\rVert$. Tối đa margin ⇔ **tối thiểu** $\lVert\mathbf w\rVert$:

$$\boxed{\min_{\mathbf w, b}\ \frac12\lVert\mathbf w\rVert^2 \quad\text{s.t.}\quad y_i(\mathbf w^\top\mathbf x_i + b) \ge 1\ \ \forall i}$$

Đây là bài toán **quy hoạch toàn phương lồi** (hàm mục tiêu bậc hai, ràng buộc tuyến tính) → nghiệm toàn cục duy nhất.

## 3. Giải trực tiếp bằng scipy

In [ ]:
def objective(params):
    w = params[:2]
    return 0.5 * w @ w


constraints = [{"type": "ineq", "fun": lambda params, i=i: y[i] * (params[:2] @ X[i] + params[2]) - 1} for i in range(len(y))]
res = minimize(objective, x0=np.array([1.0, 1.0, 0.0]), constraints=constraints, method="SLSQP")
w_opt, b_opt = res.x[:2], res.x[2]
margins = y * (X @ w_opt + b_opt)
print(f"hội tụ: {res.success} | w = {w_opt.round(4)}, b = {b_opt:.4f} | margin = 1/‖w‖ = {1 / np.linalg.norm(w_opt):.4f}")
print(f"y·(wᵀx + b) nhỏ nhất = {margins.min():.4f} (≈ 1 ở các support vector)")
sv_mask = np.isclose(margins, 1, atol=1e-3)
print("số support vector:", sv_mask.sum())

In [ ]:
def draw_svm(ax, w, b, sv_mask, title):
    draw_points(ax)
    for offset, style in [(0, "k-"), (1, "k--"), (-1, "k--")]:
        ax.plot(xs, (offset - b - w[0] * xs) / w[1], style, lw=1.2)
    ax.scatter(X[sv_mask, 0], X[sv_mask, 1], s=220, facecolors="none", edgecolors="k", lw=1.5, label="support vectors")
    ax.set_title(title)
    ax.legend(loc="upper left", fontsize=8)


fig, ax = plt.subplots(figsize=(6.5, 5.5))
draw_svm(ax, w_opt, b_opt, sv_mask, f"Hard-margin SVM: độ rộng lề = 2/‖w‖ = {2 / np.linalg.norm(w_opt):.2f}")
plt.show()

## 4. Support vectors

Chỉ những điểm **nằm trên lề** (đường nét đứt) mới "đỡ" ranh giới — gọi là **support vectors**. Xoá bất kỳ điểm nào khác (không phải support vector) rồi train lại → ranh giới **không đổi**:

In [ ]:
keep = sv_mask | (rng.random(len(y)) < 0.3)              # giữ support vectors + 30% điểm còn lại
res2 = minimize(objective, x0=np.array([1.0, 1.0, 0.0]), method="SLSQP",
                constraints=[{"type": "ineq", "fun": lambda p, i=i: y[i] * (p[:2] @ X[i] + p[2]) - 1} for i in np.flatnonzero(keep)])
print(f"dùng {keep.sum()}/{len(y)} điểm: w = {res2.x[:2].round(4)}, b = {res2.x[2]:.4f}")
print("ranh giới giống hệt:", np.allclose(res2.x, res.x, atol=1e-3))

Tính chất **thưa** này khác hẳn Logistic Regression — nơi **mọi** điểm đều ảnh hưởng tới nghiệm (dù ít hay nhiều).

## 5. Đối chiếu với scikit-learn và Logistic Regression

`SVC(kernel="linear", C=rất lớn)` ≈ hard-margin SVM:

In [ ]:
svc = SVC(kernel="linear", C=1e6).fit(X, y)
print(f"sklearn: w = {svc.coef_[0].round(4)}, b = {svc.intercept_[0]:.4f} | support vectors: {len(svc.support_)}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
sk_sv = np.zeros(len(y), bool)
sk_sv[svc.support_] = True
draw_svm(axes[0], svc.coef_[0], svc.intercept_[0], sk_sv, "SVM (sklearn)")
lr = LogisticRegression(C=1e6, max_iter=10_000).fit(X, y)
draw_points(axes[1])
axes[1].plot(xs, -(lr.coef_[0][0] * xs + lr.intercept_[0]) / lr.coef_[0][1], "k-")
axes[1].set_title("Logistic Regression (C lớn): ranh giới khác — phụ thuộc mọi điểm")
plt.show()

## 🧠 Kiểm tra nhanh

1. Vì sao có thể chuẩn hoá $(\mathbf w, b)$ sao cho điểm gần nhất có $y_i(\mathbf w^\top\mathbf x_i + b) = 1$?
2. Margin của SVM bằng bao nhiêu theo $\mathbf w$? Vì sao tối đa margin ⇔ tối thiểu $\frac12\lVert\mathbf w\rVert^2$?
3. Điều gì xảy ra với ranh giới nếu xoá một điểm không phải support vector? Một support vector?
4. Hard-margin SVM thất bại thế nào khi dữ liệu **không** tách được tuyến tính?

## 📌 Tóm tắt

- SVM chọn siêu phẳng có margin lớn nhất: $\min \frac12\lVert\mathbf w\rVert^2$ s.t. $y_i(\mathbf w^\top\mathbf x_i + b) \ge 1$ — QP lồi.
- Chỉ support vectors quyết định nghiệm (thưa).
- Hard-margin vô nghiệm khi dữ liệu không tách được → soft margin (bài 02).

## 📚 Tài liệu tham khảo

- Cortes & Vapnik (1995) — *Support-Vector Networks*
- Vũ Hữu Tiệp — [Support Vector Machine](https://machinelearningcoban.com/2017/04/09/smv/)
- CS229 Lecture notes — Support Vector Machines